In [8]:
import pandas as pd
import lightgbm as lgb

# =========================================================
# STEP 1: Load the Trained Model
# =========================================================
print("Loading the trained LightGBM model...")
model = lgb.Booster(model_file='lightgbm_onion_1day_horizon_final.txt')

Loading the trained LightGBM model...


In [9]:
# =========================================================
# STEP 2: Get "Today's" Data Snapshot
# =========================================================
# We load the dataset to get the most recent state of the market
df = pd.read_csv('final_model_ready_pune_data_ver2.csv')
df['arrival_date'] = pd.to_datetime(df['arrival_date'])

# Find the most recent date in the dataset (This represents "Today")
latest_date = df['arrival_date'].max()

# Filter the dataset so we ONLY have the rows for "Today"
# This gives us one row per mandi-variety combination
today_df = df[df['arrival_date'] == latest_date].copy()

print(f"Current Market Date ('Today'): {latest_date.date()}")
print(f"Generating price comparison for 'Tomorrow'...\n")

Current Market Date ('Today'): 2026-02-21
Generating price comparison for 'Tomorrow'...



In [10]:
# =========================================================
# STEP 3: Format Data for Prediction
# =========================================================
# The model requires the exact same categorical data types used during training
categorical_cols = ['mandi_name', 'district', 'state', 'variety']
for col in categorical_cols:
    if col in today_df.columns:
        today_df[col] = today_df[col].astype('category')

# Drop the columns the model shouldn't see (dates, targets, text identifiers)
drop_cols = ['arrival_date', 'target_price', 'commodity']
X_today = today_df.drop(columns=drop_cols, errors='ignore')

In [11]:
# =========================================================
# STEP 4: Predict Tomorrow's Prices
# =========================================================
# Generate predictions for all mandis instantly
today_df['predicted_tomorrow_price'] = model.predict(X_today)

In [14]:
# =========================================================
# STEP 5: Build the Farmer's Report
# =========================================================
# Extract only the columns the farmer cares about
comparison_report = today_df[['mandi_name', 'variety', 'latest_date','modal_price', 'predicted_tomorrow_price']].copy()

# Rename columns for clarity
comparison_report.rename(columns={
    'date_today' : 'latest_date',
    'modal_price': 'price_today',
    'predicted_tomorrow_price': 'predicted_price_tomorrow'
}, inplace=True)

# Calculate the expected day-over-day change (Trend)
comparison_report['expected_change'] = comparison_report['predicted_price_tomorrow'] - comparison_report['price_today']

# Sort from HIGHEST predicted price to LOWEST
comparison_report = comparison_report.sort_values(by='predicted_price_tomorrow', ascending=False).reset_index(drop=True)

# Format the numbers as currency for nice terminal printing

comparison_report['price_today'] = comparison_report['price_today'].apply(lambda x: f"₹ {x:.2f}")
comparison_report['predicted_price_tomorrow'] = comparison_report['predicted_price_tomorrow'].apply(lambda x: f"₹ {x:.2f}")
comparison_report['expected_change'] = comparison_report['expected_change'].apply(lambda x: f"+₹ {x:.2f}" if x > 0 else f"-₹ {abs(x):.2f}")

# Display the final report
print("================================================================")
print("             GEO-PRICE COMPARISON REPORT FOR FARMERS            ")
print("================================================================")
print(comparison_report.to_string(index=False))
print("================================================================")

KeyError: "['latest_date'] not in index"